### Exploration Code - Police Calls San Diego

This file is to perform EDA on the police calls provided by the City of San Diego. 

The main data source is found here: https://data.sandiego.gov/datasets/police-calls-for-service/

There is no database manipulation in this file. This file is only meant for EDA purposes. 

***

In [4]:
#Libraries
import pandas as pd
from datetime import datetime
import numpy as np
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim
import requests
from tabula.io import read_pdf

### Load Calls with Addresses From 2024

Next, we begin the process of loading in the calls from this year. We start with defining a Get_Data function that will allow us to check if the data already exists when we begin loading in calls and their addresses.

In [5]:
#function to download yearly data
def Get_Data(year):
    url = f"https://seshat.datasd.org/police_calls_for_service/pd_calls_for_service_{year}_datasd.csv"
    df = pd.read_csv(url)
    return df

In [6]:
url = f"https://seshat.datasd.org/police_calls_for_service/pd_cfs_calltypes_datasd.csv"
call_type = pd.read_csv(url).dropna(axis=1)
call_type['call_length'] = call_type['call_type'].str.len()
call_type['description_length'] = call_type['description'].str.len()
call_type = call_type.sort_values(by=['description_length'])

In [7]:
url = f"http://seshat.datasd.org/pd/pd_dispo_codes_datasd.csv"
disposition = pd.read_csv(url).dropna(axis=1)
disposition['description_length'] = disposition['description'].str.len()
disposition = disposition.sort_values(by=['description_length'])

In [10]:
beats_url = f"https://seshat.datasd.org/gis_police_beats/pd_beat_codes_list_datasd.csv"
beat = pd.read_csv(beats_url).dropna(axis=1)
beat_split = beat.rename(columns={0: "beat",
                                        1: "neighborhood"})
beat_split.loc[len(beat_split.index)] = [-1, 'No Beat']
beat_split.loc[len(beat_split.index)] = [700, '700'] 
beat_split.loc[len(beat_split.index)] = [62, '62']
beat_split.loc[len(beat_split.index)] = [63, '63']
beat_split.loc[len(beat_split.index)] = [64, '64'] 
beat_split.loc[len(beat_split.index)] = [0, '0'] 
beat_split.loc[len(beat_split.index)] = [300, '300']
beat_split.loc[len(beat_split.index)] = [9, '9'] 

In [ ]:

url = f"http://seshat.datasd.org/pd/pd_cfs_priority_defs_datasd.pdf"
priority = read_pdf(url)
#priority

In [ ]:
#Current year
current_year = datetime.now().year

In [ ]:
#Current Data
current_data = Get_Data(current_year)

#Last Year
last_year_data = Get_Data(current_year-1)

#Year - 2
yearM2_data = Get_Data(current_year-2)

# current_data.tail()

# EDA

In [ ]:
current_data.describe()

In [ ]:
last_year_data.info()

In [ ]:
#Expore Data
def Explore_Data(df):
    #Clean Data
    df = df.dropna(how='all', axis=1)
    
    #Initial table
    freqDF = pd.DataFrame(columns=['Feature',
                                   'Mode',
                                   'Mode Freq.',
                                   'Mode %',
                                   '2nd Mode',
                                   '2nd Mode Freq.',
                                   '2nd Mode %'])
    for col in df.columns:
        freq = df[col].value_counts()
        freqdf = freq.to_frame()
        fRow = freqdf.iloc[0]
        secRow = freqdf.iloc[1]
        fPrct = fRow[0] / len(df[col])
        secPrct = secRow[0] / len(df[col])
        try:
            mode1 = int(fRow.name)
        except:
            mode1 = fRow.name
        try:
            mode2 = int(secRow.name)
        except:
            mode2 = secRow.name
        data = {'Feature':col,
                'Mode':mode1,
                'Mode Freq.':fRow[0],
                'Mode %':fPrct,\
                '2nd Mode':mode2,
                '2nd Mode Freq.':secRow[0],
                '2nd Mode %':secPrct}
        freqDF.loc[len(freqDF)] = data

    freqDF = freqDF.set_index('Feature')

    #Nulls, Counts, Cardinality
    NUllFeatures = round(df.isnull().sum() / df.shape[0],4)\
          .sort_values(ascending=False)
    Count = df.count()
    uni = df.nunique()

    #Formating
    NUllFeatures.to_frame(name="% Miss.")
    Count.to_frame(name="Count")
    uni.to_frame()
    result = pd.concat([Count, NUllFeatures,uni], axis=1)
    result.columns =["Count","% Miss.","Card."]
    result = pd.concat([result, freqDF], axis=1)
    result = result.style.format({'% Miss.': "{:.1%}",
                         'Mode %': "{:.0%}",
                         '2nd Mode %': "{:.0%}",
                         'Count': "{:,}",
                         'Card.': "{:,}",
                         'Mode Freq.': "{:,}",
                        '2nd Mode Freq.': "{:,}"})
    return result

In [ ]:
Explore_Data(current_data)

In [ ]:
Explore_Date(last_year_data)

In [ ]:
Explore_Date(yearM2_data)

# Data Prep

In [ ]:
def AddressNumberStr(df):
    df['address_number_primary_str'] = df['address_number_primary'].astype(str)
    df.address_number_primary_str.replace('0', np.nan, inplace=True)
    return df

In [ ]:
def AddressField(df,City,State):
    df['Address'] = df[['address_number_primary_str',
                        'address_dir_primary',
                        'address_road_primary',
                        'address_sfx_primary']].apply(lambda x: ' '.join(x.dropna()), axis=1)
    df['Address'] = df['Address'] + ' ' + City +', ' + State
    return df

In [ ]:
Address_Data = AddressNumberStr(current_data)
Address_Data = AddressField(current_data,'San Diego','California')

In [ ]:
Address_Data.tail()

# Geo Location Data Google Lat and Long

In [ ]:
def Lat_Long(address):
    lat, lng, zipcode  = None, None, None
    api_key = GOOGLE_API_KEY
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    endpoint = f"{base_url}?address={address}&key={api_key}"
    r = requests.get(endpoint)
    if r.status_code not in range(200, 299):
        #error
        return None, None
    try:
        #found
        results = r.json()['results'][0]
        lat = results['geometry']['location']['lat']
        lng = results['geometry']['location']['lng']
        zipcode = results['address_components'][-1]['long_name']
        
    except:
        pass
    return lat, lng, zipcode

def DF_GeoCode(row):
    column_name = 'Address'
    address_value = row[column_name]
    address_lat, address_lng, address_zip = Lat_Long(address_value)
    row['lat'] = address_lat
    row['lng'] = address_lng
    row['zipcode'] = address_zip
    
    return row

In [ ]:
Last2000 = Address_Data.tail(10000)
Last10 = Address_Data.tail(10)

In [ ]:
Last10_Geo = Last10.apply(DF_GeoCode, axis=1)

In [ ]:
Last10_Geo

In [ ]:
Last10.to_csv('Last10.csv')

In [ ]:
Last2000_Geo = Last2000.apply(DF_GeoCode, axis=1)

In [ ]:
Last2000_Geo.to_csv('Last2000_Geo.csv')

In [ ]:
Last2000_Geo

# Geo Location Data Zipcode

In [ ]:
def get_zipcode(df, geolocator, lat_field, lon_field):
    try:
        location = geolocator.reverse((df[lat_field], df[lon_field]))
        result = location.raw['address']['postcode']
    except:
        result = None
    return result


In [ ]:
zipcodes = Last10_Geo.apply(get_zipcode,
                            axis=1,
                            geolocator=locator,
                            lat_field='lat',
                            lon_field='lng'
                           )

In [ ]:
zipcodes

In [ ]:
Last2000_Geo['Zipcodes'] = Last2000_Geo.apply(get_zipcode,
                            axis=1,
                            geolocator=locator,
                            lat_field='lat',
                            lon_field='lng'
                           )

In [ ]:
Last2000_Geo

In [ ]:
Last2000_Geo.to_csv('Last2000_Geo.csv')

In [ ]:
test = pd.read_csv('Last2000_Geo.csv')
test

In [ ]:
#Confirm Address Length
test['Address_length'] = test['Address'].str.len()
test = test.sort_values(by=['Address_length'])
test